In [ ]:
"""
OHCA Pipeline — Dual Window (24h + 72h)

Runs the full OHCA analysis pipeline for both 24h and 72h time windows.
"""
import duckdb
from pipeline_helpers import make_config, Logger
from pipeline_steps import (
    step2_build_cohort, step3_filter_ohca_icu,
    step4a_extract_vitals, step4b_block_vitals,
    step5_vitals_plots, step6_save_for_r,
    step7_trajectory_assignment, step7b_hourly_vitals_table,
    step8_trajectory_plots, step9_traj_survival_plots,
    step10_cohort_comparison, step11_table1, save_table1_outputs,
)


In [ ]:
# %% ============ SETUP ============
con = duckdb.connect()

config_init = make_config("config.json", window_hours=24)
log_init = Logger(config_init["upload_dir"] / "pipeline_log.txt")

In [ ]:
# %% ============ BUILD COHORT ============
cohort_v2 = step2_build_cohort(config_init, con, log_init)

In [ ]:
# %% ============ FILTER OHCA → ICU ============
cohort_ohca_icu = step3_filter_ohca_icu(config_init, con, log_init, cohort_v2)

In [ ]:
# %% ============ WINDOWS Data ============
for window_hours in [24, 72]:
    config = make_config("config.json", window_hours=window_hours)
    log = Logger(config["upload_dir"] / "pipeline_log.txt")

    log.info(f"\n{'#'*60}")
    log.info(f"  WINDOW: {window_hours}h")
    log.info(f"{'#'*60}")

    # 4a: Raw vitals + temperature
    raw_vitals, raw_temp = step4a_extract_vitals(config, con, log)

    # 4b: Block vitals
    block_vitals = step4b_block_vitals(config, con, log)

    # 5: Plots
    step5_vitals_plots(config, log, block_vitals, raw_vitals)

    # 6: Save for R
    step6_save_for_r(config, log, cohort_ohca_icu, raw_vitals, raw_temp, block_vitals)

    # 7: Trajectory assignment
    traj_assignment, vitals_temp = step7_trajectory_assignment(config, log, cohort_ohca_icu, raw_temp)
                                                
    # 7b: Hourly vitals table
    hourly_wide = step7b_hourly_vitals_table(config, con, log, traj_assignment)

    # 8: Trajectory plots
    vitals_by_traj = step8_trajectory_plots(config, log, vitals_temp, traj_assignment)

    # 9: Trajectory × survival plots
    step9_traj_survival_plots(config, log, vitals_by_traj, cohort_ohca_icu, traj_assignment)

    # 10: Cohort comparison
    step10_cohort_comparison(config, log, cohort_ohca_icu, cohort_v2)

    # 11: Table 1
    table1 = step11_table1(config, con, log, traj_assignment)
    save_table1_outputs(config, log, table1)

    log.info(f"\n  WINDOW {window_hours}h COMPLETE ✓")

print("\n" + "="*60)
print("  ALL WINDOWS COMPLETE")
print("="*60)